In [1]:
import pandas as pd
import numpy as np
import shutil
import os
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer, 
    DataCollatorWithPadding, 
    EarlyStoppingCallback
)

In [ ]:
configuraciones = [
    {"nombre": "prep_3_len_768", "archivo": "../Data/interim/tcga_simple_train_preprocessed_3.csv", "max_len": 768},
    {"nombre": "prep_3_len_1024", "archivo": "../Data/interim/tcga_simple_train_preprocessed_3.csv", "max_len": 1024},
    {"nombre": "prep_3_len_1280", "archivo": "../Data/interim/tcga_simple_train_preprocessed_3.csv", "max_len": 1280},
    {"nombre": "prep_3_len_1536", "archivo": "../Data/interim/tcga_simple_train_preprocessed_3.csv", "max_len": 1536},
    {"nombre": "prep_3_aug_len_768", "archivo": "../Data/interim/tcga_simple_train_preprocessed_3_augmented.csv", "max_len": 768},
    {"nombre": "prep_3_aug_len_1024", "archivo": "../Data/interim/tcga_simple_train_preprocessed_3_augmented.csv", "max_len": 1024},
    {"nombre": "prep_3_aug_len_1280", "archivo": "../Data/interim/tcga_simple_train_preprocessed_3_augmented.csv", "max_len": 1280},
    {"nombre": "prep_3_aug_len_1536", "archivo": "../Data/interim/tcga_simple_train_preprocessed_3_augmented.csv", "max_len": 1536}
]

In [3]:
label_mapping = {"T1": 0, "T2": 1, "T3": 2, "T4": 3}
tokenizer = AutoTokenizer.from_pretrained("yikuan8/Clinical-Longformer")

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    return {'macro_f1': f1_score(labels, preds, average='macro')}

resultados_finales = []

In [7]:
for conf in configuraciones:
    print("\n" + "="*50)
    print(f"TRABAJANDO EN: {conf['nombre']}")
    print("="*50)

    df = pd.read_csv(conf['archivo'])
    X_train, X_test, y_train, y_test = train_test_split(df['text'], df['t'], test_size=0.2, random_state=23, stratify=df['t'])
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=33, stratify=y_train)

    train_ds = Dataset.from_dict({"text": X_train.tolist(), "labels": [label_mapping[label] for label in y_train.tolist()]})
    val_ds = Dataset.from_dict({"text": X_val.tolist(), "labels": [label_mapping[label] for label in y_val.tolist()]})
    test_ds = Dataset.from_dict({"text": X_test.tolist(), "labels": [label_mapping[label] for label in y_test.tolist()]})

    def tokenize_fn(df):
        return tokenizer(df["text"], padding="max_length", truncation=True, max_length=conf['max_len'])

    train_tkn = train_ds.map(tokenize_fn, batched=True)
    val_tkn = val_ds.map(tokenize_fn, batched=True)
    test_tkn = test_ds.map(tokenize_fn, batched=True)

    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)
    weights = compute_class_weight("balanced", classes=np.unique(y_train_encoded), y=y_train_encoded)
    weights_tensor = torch.tensor(weights, dtype=torch.float)

    model = AutoModelForSequenceClassification.from_pretrained("yikuan8/Clinical-Longformer", num_labels=4)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)

    for param in model.parameters():
      param.requires_grad = False

    # 2. Descongelamos la cabeza de clasificación de Longformer
    # En SequenceClassification de Hugging Face, la cabeza suele llamarse 'classifier'
    for param in model.classifier.parameters():
        param.requires_grad = True

    # 3. Descongelamos las últimas capas del encoder de Longformer
    capas = model.longformer.encoder.layer
    num_capas_a_entrenar = 3

    for i in range(len(capas) - num_capas_a_entrenar, len(capas)):
        for param in capas[i].parameters():
            param.requires_grad = True

    model.loss_function = torch.nn.CrossEntropyLoss(weight=weights_tensor.to(device))

    training_args = TrainingArguments(
        output_dir=f"../Models/longformer-{conf['nombre']}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=5e-5,
        per_device_train_batch_size=6,
        num_train_epochs=5,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        fp16=True if torch.cuda.is_available() else False,
        logging_steps=200,
        report_to="none",
        optim="adamw_torch_fused"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tkn,
        eval_dataset=val_tkn,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
    )

    trainer.train()

    metrics = trainer.evaluate(test_tkn)
    resultados_finales.append({
        "Escenario": conf['nombre'],
        "Longitud": conf['max_len'],
        "Macro-F1": metrics['eval_macro_f1']
    })

    del model
    del trainer
    shutil.rmtree("../Models")
    os.makedirs("../Models")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


TRABAJANDO EN: prep_3_len_1536


Map:   0%|          | 0/4144 [00:00<?, ? examples/s]

Map:   0%|          | 0/1037 [00:00<?, ? examples/s]

Map:   0%|          | 0/1296 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/269 [00:00<?, ?it/s]

[transformers] LongformerForSequenceClassification LOAD REPORT from: yikuan8/Clinical-Longformer
Key                                | Status     | 
-----------------------------------+------------+-
lm_head.dense.weight               | UNEXPECTED | 
lm_head.dense.bias                 | UNEXPECTED | 
lm_head.bias                       | UNEXPECTED | 
lm_head.decoder.weight             | UNEXPECTED | 
lm_head.decoder.bias               | UNEXPECTED | 
lm_head.layer_norm.weight          | UNEXPECTED | 
longformer.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias            | UNEXPECTED | 
classifier.out_proj.bias           | MISSING    | 
classifier.dense.bias              | MISSING    | 
classifier.out_proj.weight         | MISSING    | 
classifier.dense.weight            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from 

KeyboardInterrupt: 

In [ ]:
df_results = pd.DataFrame(resultados_finales)
print("\n--- COMPARATIVA FINAL ---")
print(df_results)